# New decision models: first contact

Tries the two hosted decision APIs we can reach now, **Cloudflare Clef / Clef-flash** and **Perplexity pplx-decider-v1-27b**, in three steps:

1. **Vendor example.** Send each vendor's own documented example request and look at the raw response, so we know the real shape before trusting our adapter.
2. **Our adapter.** Send the same request through `goldrails_bench.hosted` and check it parses the answers.
3. **Edition 2 smoke.** Run `benchmark/runs/e2_smoke.py` for these systems: 20 public **tune** rows per subtask, never test or unpublished rows.

Keys come from `.env` (`CLOUDFLARE_ACCOUNT_ID`, `CLOUDFLARE_API_TOKEN` or `CLOUDFLARE_AUTH_TOKEN`, `PERPLEXITY_API_KEY`). A system with no key is skipped. Nothing here prints a key.

Before sending anything beyond public tune rows to Perplexity, get written confirmation of Decisions API data retention (see `docs/benchmark/28-new-decision-models-research.md`). Keep Cloudflare AI Gateway logging off.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

import httpx
import pandas as pd
from dotenv import find_dotenv, load_dotenv

REPO = Path.cwd()
while not (REPO / "benchmark").is_dir():
    REPO = REPO.parent
load_dotenv(find_dotenv(usecwd=True))
sys.path[:0] = [str(REPO / "benchmark"), str(REPO / "dataset")]

CF_ACCOUNT = os.environ.get("CLOUDFLARE_ACCOUNT_ID") or ""
CF_TOKEN = os.environ.get("CLOUDFLARE_API_TOKEN") or os.environ.get("CLOUDFLARE_AUTH_TOKEN") or ""
PPLX_KEY = os.environ.get("PERPLEXITY_API_KEY") or ""
HAVE = {"cloudflare": bool(CF_ACCOUNT and CF_TOKEN), "perplexity": bool(PPLX_KEY)}
pd.DataFrame([{"vendor": k, "key set": v} for k, v in HAVE.items()])

## 1. Vendor examples, raw responses

In [ ]:
def show(resp):
    hdr = {k: v for k, v in resp.headers.items() if k.lower().startswith(("x-request-id", "x-ratelimit", "cf-ray"))}
    print("HTTP", resp.status_code, hdr)
    try:
        print(json.dumps(resp.json(), indent=1)[:3000])
    except ValueError:
        print(resp.text[:1000])

CF_EXAMPLE = {
    "state": "Checkout has been failing for every customer for the last hour.",
    "questions": {
        "urgent": {"type": "noul", "instructions": "Is this support request urgent?"},
        "team": {"type": "choice", "instructions": "Which team should handle this request?",
                 "criteria": {"billing": "Payments, invoices, and refunds",
                              "technical": "Outages, errors, and configuration",
                              "sales": "Plans and upgrades"}},
        "severity": {"type": "score", "instructions": "How severe is the customer impact?",
                     "criteria": ["No impact", "Minor", "Major", "Critical"]},
    },
}
cf_raw = {}
if HAVE["cloudflare"]:
    for model in ("clef", "clef-flash"):
        url = f"https://api.cloudflare.com/client/v4/accounts/{CF_ACCOUNT}/ai/run/@cf/cloudflare/{model}"
        r = httpx.post(url, headers={"Authorization": f"Bearer {CF_TOKEN}"}, json={"model": model, **CF_EXAMPLE}, timeout=60)
        print(f"--- {model}"); show(r); cf_raw[model] = r
else:
    print("Cloudflare skipped: set CLOUDFLARE_ACCOUNT_ID and CLOUDFLARE_API_TOKEN in .env")

In [ ]:
PPLX_EXAMPLE = {
    "model": "pplx-decider-v1-27b",
    "state": {"title": "Battery died after two weeks",
              "review": "The headphones sound great, but the battery stopped charging after two weeks."},
    "questions": {
        "defect": {"type": "noul", "instructions": "Does the review report a product defect?"},
        "sentiment": {"type": "choice", "instructions": "What is the overall sentiment of the review?",
                      "criteria": {"positive": "Mostly satisfied", "mixed": "Praise and complaints in one review",
                                   "negative": "Mostly dissatisfied"}},
        "severity": {"type": "score", "instructions": "How severe is the reported problem?",
                     "criteria": ["Cosmetic", "Inconvenient", "Product unusable"]},
    },
}
pplx_raw = None
if HAVE["perplexity"]:
    pplx_raw = httpx.post("https://api.perplexity.ai/v1/decisions", headers={"Authorization": f"Bearer {PPLX_KEY}"},
                          json=PPLX_EXAMPLE, timeout=30)
    show(pplx_raw)
    print("Perplexity's documented answer: defect ~0.94, sentiment 'mixed', severity ~1.78")
else:
    print("Perplexity skipped: set PERPLEXITY_API_KEY in .env")

## 2. The same requests through our adapter clients

In [ ]:
from goldrails_bench import hosted

rows = []
clients = []
if HAVE["cloudflare"]:
    clients += [(m, hosted.CloudflareSystemOneClient.from_env(m), CF_EXAMPLE) for m in ("clef", "clef-flash")]
if HAVE["perplexity"]:
    clients += [("perplexity", hosted.PerplexityDecisionsClient.from_env(), PPLX_EXAMPLE)]
for name, client, ex in clients:
    call = client.ask(ex["state"], ex["questions"])
    rows.append({"system": name, "ok": call.ok, "error": call.error, "model": call.model,
                 "latency_s": round(call.latency_s or 0, 3), "input_tokens": (call.usage or {}).get("input_tokens"),
                 "answers": json.dumps(call.answers)[:300] if call.answers else None})
pd.DataFrame(rows) if rows else "no clients configured"

## 3. Edition 2 smoke: 20 public tune rows per subtask

Runs the same script the other systems used. It skips rows already recorded, so rerunning is cheap. Results land in `benchmark/results/edition2-smoke/` (ids and model outputs only, no row text).

In [ ]:
systems = [s for s, ok in (("clef", HAVE["cloudflare"]), ("clef-flash", HAVE["cloudflare"]), ("perplexity", HAVE["perplexity"])) if ok]
if systems:
    for stage in (["run", "--systems", ",".join(systems)], ["report"]):
        p = subprocess.run([sys.executable, str(REPO / "benchmark/runs/e2_smoke.py"), *stage],
                           cwd=REPO, capture_output=True, text=True)
        print(p.stdout[-4000:], p.stderr[-2000:])
else:
    print("nothing to run: no keys set")

In [ ]:
summary = json.loads((REPO / "benchmark/results/edition2-smoke/summary.json").read_text())
df = pd.DataFrame(summary["table"])
cols = ["system", "suite", "subtask", "rows", "decided", "failed", "not_offered", "correct", "truncated",
        "max_usage_input_tokens", "p50_latency_s"]
new = df[df.system.str.contains("clef|pplx|perplexity", case=False)]
(new if len(new) else df)[cols].sort_values(["system", "suite", "subtask"]).reset_index(drop=True)

## What to check

- Every row **decided**, 0 failed, probabilities in [0, 1].
- `correct` is a sanity signal only (20 tune rows); the benchmark score comes from the full run.
- Clef-flash on grounding: Cloudflare's own numbers show it weak on RAGTruth, so a low count there is expected.
- Perplexity: the response `model` should be `pplx-decider-v1-27b`; the adapter refuses any Jev name.